# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Get a summary from metadata
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n")
print(f"Dataset Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"DOI/Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print("Keywords:", getattr(metadata, 'keywords', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their `@id`s from the Croissant schema.

In Croissant datasets, a RecordSet represents a table-like structure. We'll list all available record sets, their fields, and the associated column `@id`s for reference.

In [ ]:
# Get all record sets in this dataset
record_sets = dataset.metadata.record_sets
print(f"Number of RecordSets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"- RecordSet name: {rs.name} | @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print(f"  Fields in this RecordSet:")
    for field in rs.fields:
        print(f"    - Field: {field.name} | @id: {field.id} | dataType: {field.data_type}")
        if getattr(field, 'column', None):
            if isinstance(field.column, list):
                columns = [c.id for c in field.column]
            else:
                columns = [field.column.id]
            print(f"      columns: {columns}")
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

Below, we show how to load all records for each RecordSet into separate pandas DataFrames, indexed by their `@id`.

In [ ]:
# For demonstration, gather all available RecordSet @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Extract the first record set as primary (for exploration)
primary_rs_id = record_set_ids[0]

# Load each record set into its own DataFrame
for rs_id in record_set_ids:
    print(f"Loading records for RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"  -> Loaded {len(df)} records. Columns: {df.columns.tolist()}\n")

# Display the columns and the first 5 rows of the primary DataFrame
print(f"Primary RecordSet: {primary_rs_id}\nColumns: {dataframes[primary_rs_id].columns.tolist()}")
dataframes[primary_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

We will select a numeric field, filter records, normalize the data, and show some basic grouping.

In [ ]:
# Find the field and column ids for numeric analysis from the primary record set
primary_record_set = next(rs for rs in dataset.metadata.record_sets if rs.id == primary_rs_id)
numeric_fields = [field for field in primary_record_set.fields if field.data_type in ('Float','Integer','Number') or 'float' in str(field.data_type).lower() or 'number' in str(field.data_type).lower()]

print("Numeric fields available:")
for f in numeric_fields:
    print(f"- {f.name} (@id: {f.id}, datatype: {f.data_type})")

# We'll try to select the first numeric field for demonstration
if numeric_fields:
    numeric_field = numeric_fields[0].id
else:
    print("No numeric fields found.")
    numeric_field = None

# Display the first 5 values if possible
if numeric_field:
    df = dataframes[primary_rs_id]
    print(f"Sample values for {numeric_field}:")
    print(df[numeric_field].head())
    # Try to coerce to float, if it's not already
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Simple EDA: filter, normalize, group
    threshold = df[numeric_field].median()  # Use median as demo threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"\nFiltered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Find a potential grouping field (categorical)
    categorical_fields = [f for f in primary_record_set.fields if f not in numeric_fields and f.data_type in ('Text', 'String') or 'str' in str(f.data_type).lower()]
    if categorical_fields:
        group_field = categorical_fields[0].id
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean {numeric_field} by {group_field} (top 5):")
            print(grouped_df.head())
    else:
        group_field = None
else:
    print('No numeric fields found for filtering and normalization.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we plot the distribution of a numeric field (if found).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we have loaded the FAIR^2 dataset for mass spectrometry analysis of extracellular vesicles, explored the schema, extracted and displayed data using specific `@id` references, and performed basic exploratory data analysis and visualization using `mlcroissant`.

- We identified available RecordSets, fields, and their unique IDs as per the Croissant specification.
- We demonstrated filtering, normalization, and grouping of numeric fields.
- Visual explorations highlight data distributions and group statistics, which can serve for further scientific analysis.

Continue by further cleaning, feature engineering, or model building as appropriate for your application.